In [ ]:
!pwd

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("AAI_API_KEY")

In [ ]:
import assemblyai as aai
aai.settings.api_key = api_key

In [ ]:
from pathlib import Path
import requests
import time

base_dir = Path.cwd()

transaction_dir = base_dir / "Unclipped Processed Transactions"

In [ ]:
files = [f for f in transaction_dir.iterdir()
               if f.is_file() and not f.name.startswith(".")]

In [ ]:
base_url = "https://api.assemblyai.com"

headers = {
"authorization": api_key
}

In [ ]:
#Script to collect transcript with timestamps and operator/customer splits for julian files
transcripts = {}

for file in files:
    with open(file, "rb") as f:
        response = requests.post(base_url + "/v2/upload", headers=headers, data=f)
        if response.status_code != 200:
            print(f"Error: {response.status_code}, Response: {response.text}")
            response.raise_for_status()
        upload_json = response.json()
        audio_file = upload_json["upload_url"]


    data = {
    "audio_url": audio_file,
    "speech_models": ["universal-3-pro", "universal-2"],
    "language_detection": True,
    "speaker_labels": True,
    "speakers_expected": 2,
    "speech_understanding": {
        "request": {
            "speaker_identification": {
                "speaker_type": "name",
                "known_values": ["Customer", "Operator"]
                }
            }
        }
    }
    
    response = requests.post(base_url + "/v2/transcript", headers=headers, json=data)
    transcript_id = response.json()["id"]
    polling_endpoint = base_url + f"/v2/transcript/{transcript_id}"

    while True:
        transcript = requests.get(polling_endpoint, headers=headers).json()
        if transcript["status"] == "completed":
            break
        elif transcript["status"] == "error":
            raise RuntimeError(f"Transcription failed: {transcript['error']}")
        else:
            time.sleep(3)

    transcripts[file] = transcript

In [ ]:
intervals = {}

for key, transcript in transcripts.items():
    specific_intervals = []
    for utterance in transcript["utterances"]:
        if utterance["speaker"] == "Operator":
            specific_intervals.append((utterance["start"], utterance["end"]))
    intervals[key] = specific_intervals

In [ ]:
from pathlib import Path
from pydub import AudioSegment

def extract_and_concat_segments(
    input_path: str | Path,
    segments_ms: list[tuple[int, int]],
    output_path: str | Path,
    output_format: str | None = None,  # e.g. "wav", "mp3"; if None, infer from output_path suffix
) -> Path:
    input_path = Path(input_path)
    output_path = Path(output_path)

    if output_format is None:
        output_format = output_path.suffix.lstrip(".").lower() or "wav"

    audio = AudioSegment.from_file(input_path)

    # Build output by concatenating slices
    out = AudioSegment.silent(duration=0, frame_rate=audio.frame_rate)

    for start_ms, end_ms in segments_ms:
        if start_ms < 0 or end_ms < 0:
            raise ValueError(f"Negative times not allowed: {(start_ms, end_ms)}")
        if end_ms <= start_ms:
            continue  # skip empty/invalid segment quietly

        start_ms = min(start_ms, len(audio))
        end_ms = min(end_ms, len(audio))
        if end_ms <= start_ms:
            continue

        out += audio[start_ms:end_ms]

    out.export(output_path, format=output_format)
    return output_path

# Example:
# segments = [(1200, 4500), (9000, 12500), (20000, 23000)]
# extract_and_concat_segments("input.wav", segments, "output.wav")


In [ ]:
output_dir = base_dir / "Concat Transactions"
output_dir.mkdir(parents = True, exist_ok = True)

In [ ]:
file_name_counter = 1
for file, specific_intervals in intervals.items():
    operator_name = file.stem.split("_tx_")[0]
    extract_and_concat_segments(file, specific_intervals, output_dir / f"{operator_name}_concat_{file_name_counter}.wav")
    file_name_counter += 1


In [ ]:
from pathlib import Path
base_dir = Path.cwd()

#Set up Various Directories
#Julian and George Raw Ground Truths Live in "Raw Ground Truths"
#They will go through processing and move to Processed Ground Truths, then prepare_audio_for_embedding() method and move to "To Be Embedded Ground Truths"
#Transaction Clips live in "Processed Transaction Clips". They only need to go through prepare_audio_for_embedding() method and move to "To Be Embedded Transactions


raw_truth_dir = base_dir / "Raw Ground Truths"
transaction_dir = base_dir / "Concat Transactions"

processed_dir = base_dir / "Processed Ground Truths"
output_dir = base_dir / "To Be Embedded Ground Truths"
to_embed_dir = base_dir / "To Be Embedded Transactions"

processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents = True, exist_ok = True)
to_embed_dir.mkdir(parents = True, exist_ok=True)

In [ ]:
#Import Methods from utils.py
from utils import prepare_audio_for_embedding, decode_audio, preprocess_audio, save_audio

In [ ]:
#Preprocessing Steps for Raw Ground Truths
#Pass through methods decode_audio(), preprocess_audio(), and save_audio()
#decode_audio() and preprocess_audio() combine to apply bandpass, AGC and limiter (in that order)
#save_audio() will save ground truths to "Processed Ground Truths"
from tqdm import tqdm

raw_input_files = [
    p for p in raw_truth_dir.iterdir()
    if p.is_file() and not p.name.startswith(".")
]
print(f"Found {len(raw_input_files)} raw input files.")

processed_audio_samples = []

for i, file_path in enumerate(raw_input_files):
    file_name = str(raw_input_files[i])[len(str(raw_truth_dir))+1:]
    audio, sr = decode_audio(file_path)
    processed_audio_sample = preprocess_audio(audio, sr)
    output_path = save_audio(audio, sr, processed_dir / f"Processed_{file_name}")
    print(f"Saved file to {output_path}")

In [ ]:
#Get Preprocessed Ground Truths Ready for Embedding
#prepare_audio_for_embedding() method will set frame rate of the .wav to 16kHz
processed_files = [f for f in processed_dir.rglob("*") if f.is_file() and not f.name.startswith(".")]

print(f"Found {len(processed_files)} audio files")

for processed_file in tqdm(processed_files):
    processed_path = prepare_audio_for_embedding(processed_file, output_dir)

In [ ]:
#Get Preprocessed Transaction Clips Ready for Embedding
processed_files = [f for f in transaction_dir.rglob("*") if f.is_file() and not f.name.startswith(".")]

print(f"Found {len(processed_files)} audio files")

for processed_file in tqdm(processed_files):
    processed_path = prepare_audio_for_embedding(processed_file, to_embed_dir)

In [ ]:
#Standard Code to set up TitaNet
#Set to eval() mode for future embedding
import torch
import nemo.collections.asr as nemo_asr

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

# Pretrained TitaNet Large speaker verification model
speaker_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained(
    model_name="nvidia/speakerverification_en_titanet_large"
).to(device)

speaker_model.eval()

In [ ]:
#Process for Embedding Ground Truth Embeddings
#They are saved to Ground Truth Embeddings/

emb_dir = base_dir / "Ground Truth Embeddings"
emb_dir.mkdir(parents = True, exist_ok = True)

wav_files = sorted([p for p in output_dir.rglob("*.wav")])
print("WAV files found:", len(wav_files))
print("example:", wav_files[0] if wav_files else "None")

import numpy as np
import pandas as pd
from tqdm import tqdm

rows = []
all_embs = {}  # {relative_path: embedding_vector}

with torch.no_grad():
    for wav_path in tqdm(wav_files):
        # NeMo provides get_embedding(path_to_wav) for speaker embeddings :contentReference[oaicite:3]{index=3}
        emb = speaker_model.get_embedding(str(wav_path))

        # Normalize output type -> 1D numpy array
        if isinstance(emb, torch.Tensor):
            emb_np = emb.detach().cpu().numpy()
        else:
            emb_np = np.asarray(emb)

        emb_np = np.squeeze(emb_np)  # ensure shape (D,)

        # Save one embedding per file
        rel = wav_path.relative_to(output_dir)
        out_path = emb_dir / rel.with_suffix(".npy")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb_np)

        rows.append({
            "wav_path": str(wav_path),
            "embedding_path": str(out_path),
            "dim": int(emb_np.shape[0]),
        })
        all_embs[str(rel)] = emb_np

df = pd.DataFrame(rows)
df.to_csv(emb_dir / "index.csv", index=False)

# Also save a single archive for easy loading later
np.savez_compressed(emb_dir / "embeddings.npz", **all_embs)

print("Saved embeddings to:", emb_dir)
print(df.head())

In [ ]:
#Process for Embedding Transaction
#Saved to Transaction Embeddings/
emb_dir = base_dir / "Transaction Embeddings"
emb_dir.mkdir(parents = True, exist_ok = True)

wav_files = sorted([p for p in to_embed_dir.rglob("*.wav")])
print("WAV files found:", len(wav_files))
print("example:", wav_files[0] if wav_files else "None")

import numpy as np
import pandas as pd
from tqdm import tqdm

rows = []
all_embs = {}  # {relative_path: embedding_vector}

with torch.no_grad():
    for wav_path in tqdm(wav_files):
        # NeMo provides get_embedding(path_to_wav) for speaker embeddings :contentReference[oaicite:3]{index=3}
        emb = speaker_model.get_embedding(str(wav_path))

        # Normalize output type -> 1D numpy array
        if isinstance(emb, torch.Tensor):
            emb_np = emb.detach().cpu().numpy()
        else:
            emb_np = np.asarray(emb)

        emb_np = np.squeeze(emb_np)  # ensure shape (D,)

        # Save one embedding per file
        rel = wav_path.relative_to(to_embed_dir)
        out_path = emb_dir / rel.with_suffix(".npy")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb_np)

        rows.append({
            "wav_path": str(wav_path),
            "embedding_path": str(out_path),
            "dim": int(emb_np.shape[0]),
        })
        all_embs[str(rel)] = emb_np

df = pd.DataFrame(rows)
df.to_csv(emb_dir / "index.csv", index=False)

# Also save a single archive for easy loading later
np.savez_compressed(emb_dir / "embeddings.npz", **all_embs)

print("Saved embeddings to:", emb_dir)
print(df.head())

In [ ]:
import soundfile as sf

for wav_path in wav_files:
    info = sf.info(str(wav_path))
    if info.frames == 0:
        print("EMPTY:", wav_path, info)
        continue

    try:
        emb = speaker_model.get_embedding(str(wav_path))
    except Exception as e:
        print("FAILED:", wav_path, info, "err:", e)
        raise

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- paths (relative to current working directory) ---
tx_dir = Path.cwd() / "Transaction Embeddings"
gt_dir = Path.cwd() / "Ground Truth Embeddings"

# --- collect only .npy files, ignore everything else ---
tx_files = sorted([p for p in tx_dir.iterdir() if p.is_file() and p.suffix == ".npy"])
gt_files = sorted([p for p in gt_dir.iterdir() if p.is_file() and p.suffix == ".npy"])

if not tx_files:
    raise FileNotFoundError(f"No .npy files found in: {tx_dir}")
if not gt_files:
    raise FileNotFoundError(f"No .npy files found in: {gt_dir}")

# --- load helpers ---
def load_vec_192(path: Path) -> np.ndarray:
    """Load an embedding and return shape (192,) float32."""
    v = np.load(path)
    v = np.asarray(v, dtype=np.float32).reshape(-1)  # flatten any (1,192) or (192,1) etc.
    if v.shape[0] != 192:
        raise ValueError(f"{path.name}: expected 192 values, got {v.shape[0]} with original shape {np.load(path).shape}")
    return v

# --- load all embeddings ---
tx_mat = np.stack([load_vec_192(p) for p in tx_files], axis=0)   # (N, 192)
gt_mat = np.stack([load_vec_192(p) for p in gt_files], axis=0)   # (M, 192)

# --- cosine similarity (manual; no sklearn needed) ---
# sim = (A @ B.T) / (||A|| * ||B||)
tx_norm = np.linalg.norm(tx_mat, axis=1, keepdims=True)          # (N, 1)
gt_norm = np.linalg.norm(gt_mat, axis=1, keepdims=True)          # (M, 1)

# avoid divide-by-zero just in case
tx_norm = np.clip(tx_norm, 1e-12, None)
gt_norm = np.clip(gt_norm, 1e-12, None)

sim = (tx_mat @ gt_mat.T) / (tx_norm * gt_norm.T)               # (N, M)

# --- build dataframe ---
row_names = [p.stem for p in tx_files]
col_names = [p.stem for p in gt_files]

df = pd.DataFrame(sim, index=row_names, columns=col_names).round(5)

# mapping from full column name → operator name

operator_map = {
    'Processed_Brianna Leach-Richardson_converted': "Brianna",
       'Processed_Edward Herrera_converted': "Edward",
       'Processed_George Robbins_converted': "George",
       'Processed_Gunnar Mckinnon_converted': "Gunnar", 'Processed_Jake Wohl_converted': "Jake",
       'Processed_Julian Carmona-Munoz_converted': "Julian",
       'Processed_Katherine Campos_converted': "Katherine",
       'Processed_Katie Carbajal-Mendoza_converted': "Katie",
       'Processed_Kayla Devito_converted': "Kayla",
       'Processed_Kennith Newkirk_converted': "Kennith",
       'Processed_Kyleigh Harp_converted': "Kyleigh", 'Processed_Maria Mendoza_converted': "Maria",
       'Processed_Olivia Jensen_converted': "Olivia",
       'Processed_Oswaldo Ballesteros_converted': "Oswaldo",
       'Processed_Owen LaMontagne_converted': "Owen",
       'Processed_Shaniyah Smith_converted': "Shaniyah"
    
}





"""
operator_map = {
    "Cosine Similarity (George)": "George",
    "Cosine Similarity (Julian)": "Julian"
}
"""

# Get column name of max similarity per row
max_cols = df.idxmax(axis=1)

# Map to clean operator names
df["Recognized Operator"] = max_cols.map(operator_map)

# Convert index to a Series so we can use .str methods
index_series = df.index.to_series()

df["Ground Truth Operator"] = None  # initialize column

df.loc[index_series.str.contains("George", case=False, na=False),
       "Ground Truth Operator"] = "George"

df.loc[index_series.str.contains("Julian", case=False, na=False),
       "Ground Truth Operator"] = "Julian"

df.loc[index_series.str.contains("Jake", case=False, na=False),
       "Ground Truth Operator"] = "Jake"

df.loc[index_series.str.contains("Edward", case=False, na=False),
       "Ground Truth Operator"] = "Edward"

df["Correct Identification"] = (
    df["Recognized Operator"] == df["Ground Truth Operator"]
)

In [ ]:
df

In [ ]:
accuracy = (len(df[df["Correct Identification"] == True]) / len(df) ) * 100
print(f"Final Accuracy for binary classification task is: {accuracy}%")